# Two-Stage Learning (224px)
4 model (EfficientFormer-L1, Swin-Tiny, ConvNeXt-Tiny, CLIP ViT-B/32).
- **Stage 1:** Recyclable + Organic (224px)
- **Stage 2:** Organic + Electronic (224px)
- **Data:** backup → clean duplicate → keep leakage
- **Output:** `experiments/results/trisha_v3.0/`

In [1]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

%matplotlib inline

_cwd = Path.cwd()
if (_cwd / 'modules').exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd.parent / 'modules').exists():
    sys.path.insert(0, str(_cwd.parent))
else:
    sys.path.insert(0, str(_cwd.parent.parent))

In [2]:
try:
    import open_clip
    print('open-clip-torch already installed')
except ImportError:
    print('open-clip-torch not available - CLIP model will be skipped')
    open_clip = None

d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


open-clip-torch already installed


In [3]:
from modules.utils.config import CLASS_LABELS, IMG_SIZE, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.models.factory import TrashClassifier
from modules.models.clip_adapter import CLIPAdapter
from modules.training.loss import ClassBalancedLoss
from modules.training.train import fit
from modules.training.collator import MixUpCollator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = RESULTS / 'trisha_v3.0'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')
print(f'Project root: {PROJECT_ROOT}')

Device: cuda
Output dir: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0
Project root: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble


---
## 1. Setup Data: train_clean/
Copy `train/` →   `train_clean/`, hapus 62 duplicate, rename 3 long-path.

In [4]:
SRC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean'

if not SRC_DIR.exists():
    raise FileNotFoundError('train/ not found. Download from drive first.')

if TRAIN_DIR.exists():
    print(f'{TRAIN_DIR} already exists, skipping copy.')
else:
    print('Copying train/ -> train_clean/...')
    ret = os.system(f'robocopy "{SRC_DIR}" "{TRAIN_DIR}" /E /COPY:DAT /R:2 /W:2 /NDL /NFL /NJH /NJS')
    print('Done')

D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\data\raw\train_clean already exists, skipping copy.


In [5]:
long_names = [
    ("1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-representasi-visual-yang-luar-biasa-dari-masalah-limbah-elektronik-yang-berkembang-dan-kebutuhan-akan-solusi-daur-ulang-berkelanjutan_73523-11403.jpg", "E_001.jpg"),
    ("12.pile-discarded-motherboards-cpus-cables-disc-drives-hijacked-hardware-heap-concept-electronic-waste-tech-recycling-hardware-reuse-sustainable-technology-environmental-impact_864588-263287.jpg", "E_012.jpg"),
    ("64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat-seperti-perangkat-elektronik-yang-mengeluarkan-bahan-kimia-beracun-ke-dalam-tanah-dan-saluran-air-di-lokasi-pembuangan-ilegal_997534-43151.jpg", "E_064.jpg"),
]
def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

for old_name, new_name in long_names:
    old = TRAIN_DIR / '1_Electronic' / old_name
    new = old.with_name(new_name)
    try:
        os.rename(_safe_path(old), _safe_path(new))
        print(f'{old_name[:40]}... -> {new_name}')
    except FileNotFoundError:
        pass
    except FileExistsError:
        pass
print('Rename done.')

# Scan MD5 + remove duplicates (keep 1 per group)
hash_map = defaultdict(list)
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        try:
            with open(_safe_path(fpath), 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})
        except:
            pass

dup_groups = {h: files for h, files in hash_map.items() if len(files) > 1}
print(f'Duplicate groups: {len(dup_groups)}')

removed = 0
for h, files in dup_groups.items():
    for f in files[1:]:
        os.remove(f['path'])
        removed += 1
print(f'Removed {removed} duplicate files.')

Rename done.


Duplicate groups: 0
Removed 0 duplicate files.


---
## 2. Load & Split Data (train_clean/)

In [6]:
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean'
records = []
for label_dir in sorted(TRAIN_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    for img in sorted(label_dir.glob('*')):
        if img.is_file():
            records.append({'path': str(img.relative_to(PROJECT_ROOT)), 'label': label_dir.name})
df = pd.DataFrame(records)
print(f'Total: {len(df)}')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

Total: 26465
  0_Recyclable: 9990
  1_Electronic: 3931
  2_Organic: 12544
Train: 21172, Val: 5293


---
## 3. Dataset Class + Transforms

In [10]:
_LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}

def _open_image(path):
    safe = _safe_path(path)
    with open(safe, 'rb') as f:
        return Image.open(BytesIO(f.read())).convert('RGB')

class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = (PROJECT_ROOT / df['path']).values
        self.labels = df['label'].map(_LABEL_TO_IDX).values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = _open_image(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def make_transform(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.3, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.3),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def make_loader(df, img_size, batch_size=32, shuffle=True, use_mixup=False):
    ds = TrashDataset(df, transform=make_transform(img_size, is_train=shuffle))
    collator = MixUpCollator(alpha=0.2, num_classes=3) if (use_mixup and shuffle) else None
    sampler = None
    if shuffle:
        labels = df['label'].map(_LABEL_TO_IDX).values
        counts = torch.bincount(torch.tensor(labels))
        weights = 1.0 / counts[labels].float()
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        ds, batch_size=batch_size,
        sampler=sampler, shuffle=False if sampler else shuffle,
        num_workers=0, pin_memory=False,
        collate_fn=collator,
    )

---
## 4. Dataloaders
Stage 1: Recyclable+Organic (224px)
Stage 2: Organic+Electronic (224px)
Final eval: full 3 kelas (224px)

In [11]:
BATCH_SIZE = 32

# Stage 1: Recyclable + Organic
mask_s1 = train_df['label'].isin(['0_Recyclable', '2_Organic'])
train_s1 = train_df[mask_s1].reset_index(drop=True)
val_s1 = val_df[val_df['label'].isin(['0_Recyclable', '2_Organic'])].reset_index(drop=True)
train_loader_s1 = make_loader(train_s1, img_size=224, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s1 = make_loader(val_s1, img_size=224, shuffle=False)
print(f'Stage 1 train: {len(train_s1)}, val: {len(val_s1)}')

# Stage 2: Organic + Electronic
mask_s2 = train_df['label'].isin(['1_Electronic', '2_Organic'])
train_s2 = train_df[mask_s2].reset_index(drop=True)
val_s2 = val_df[val_df['label'].isin(['1_Electronic', '2_Organic'])].reset_index(drop=True)
train_loader_s2 = make_loader(train_s2, img_size=224, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s2 = make_loader(val_s2, img_size=224, shuffle=False)
print(f'Stage 2 train: {len(train_s2)}, val: {len(val_s2)}')

# Final full evaluation
train_loader_full = make_loader(train_df, img_size=224, use_mixup=False)
val_loader_full = make_loader(val_df, img_size=224, shuffle=False)
print(f'Full train: {len(train_df)}, val: {len(val_df)}')

Stage 1 train: 18027, val: 4507
Stage 2 train: 13180, val: 3295
Full train: 21172, val: 5293


---
## 5. Helper: move .pt to output dir

In [12]:
def move_to_outdir(name):
    for ext in ['.pt', '.json']:
        src = RESULTS / f'{name}{ext}'
        if src.exists():
            shutil.move(str(src), str(OUT_DIR / f'{name}{ext}'))

def eval_model(model, loader):
    model.to(DEVICE).eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='Eval', leave=False):
            out = model(x.to(DEVICE))
            preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(labels, preds, average='macro')
    f1_pc = f1_score(labels, preds, average=None, labels=list(range(3))).tolist()
    prec_pc = precision_score(labels, preds, average=None, labels=list(range(3))).tolist()
    rec_pc = recall_score(labels, preds, average=None, labels=list(range(3))).tolist()
    return {'f1_macro': f1, 'f1_per_class': f1_pc, 'precision': prec_pc, 'recall': rec_pc, 'preds': preds, 'labels': labels}

---
## 6. TRAIN: EfficientFormer-L1

In [13]:
MODEL = 'efficientformer_l1'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== efficientformer_l1 ===
Parameters: 11,623,355

Stage 1: Recyclable + Organic @ 224px

=== efficientformer_l1_s1: Phase 1 — Head Only ===


  E01: train_loss=0.4248  val_f1=0.9334  best=0.9334


  E02: train_loss=0.3613  val_f1=0.9375  best=0.9375


  E03: train_loss=0.3429  val_f1=0.9443  best=0.9443


  E04: train_loss=0.3501  val_f1=0.9445  best=0.9445


...

  E12: train_loss=0.1692  val_f1=0.9893  best=0.9905


  E13: train_loss=0.1669  val_f1=0.9897  best=0.9905


  E14: train_loss=0.1696  val_f1=0.9892  best=0.9905


  E15: train_loss=0.1854  val_f1=0.9897  best=0.9905
  Early stopping at epoch 15


F1 macro: 0.4826
              precision    recall  f1-score   support

0_Recyclable     1.0000    0.0015    0.0030      1998
1_Electronic     0.4579    0.9975    0.6277       786
   2_Organic     0.6951    0.9912    0.8172      2509

    accuracy                         0.6186      5293
   macro avg     0.7177    0.6634    0.4826      5293
weighted avg     0.7750    0.6186    0.4817      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\efficientformer_l1.pt/.json


In [14]:
MODEL = 'efficientformer_l1'
model = TrashClassifier(encoder_name=MODEL, num_classes=3)
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=0,
    lr_head=3e-4, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)

# Final eval ulang
model.load_state_dict(torch.load(RESULTS / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')


=== efficientformer_l1_final: Phase 1 — Head Only ===


  E01: train_loss=0.7685  val_f1=0.9498  best=0.9498


  E02: train_loss=0.1923  val_f1=0.9547  best=0.9547


  E03: train_loss=0.1807  val_f1=0.9561  best=0.9561


  E04: train_loss=0.1567  val_f1=0.9585  best=0.9585


  E05: train_loss=0.1624  val_f1=0.9567  best=0.9585


  E06: train_loss=0.1482  val_f1=0.9587  best=0.9587


  E07: train_loss=0.1447  val_f1=0.9596  best=0.9596


  E08: train_loss=0.1454  val_f1=0.9572  best=0.9596

=== efficientformer_l1_final: Phase 2 — Fine-tune All ===


F1 macro: 0.9571
              precision    recall  f1-score   support

0_Recyclable     0.9565    0.9364    0.9464      1998
1_Electronic     0.9260    0.9873    0.9557       786
   2_Organic     0.9712    0.9673    0.9692      2509

    accuracy                         0.9586      5293
   macro avg     0.9512    0.9637    0.9571      5293
weighted avg     0.9590    0.9586    0.9586      5293

Saved: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v3.0\efficientformer_l1.pt/.json


---
## 7. TRAIN: ConvNeXt-Tiny

In [ ]:
MODEL = 'convnext_tiny'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 — Recalibrate Head (full 3 classes)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate Head @ 224px (full 3 classes) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=0,
    lr_head=3e-4, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

=== convnext_tiny ===


---
## 8. TRAIN: CLIP ViT-B/32 (VLM)

In [ ]:
# Wrap CLIPAdapter to match fit() interface
class CLIPTrainable(CLIPAdapter):
    @property
    def classifier(self):
        return self.visual_projection
    @property
    def encoder(self):
        return self.clip.visual
    def freeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = True

MODEL = 'clip_vit_b32'
print(f'=== {MODEL} ===')

model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
clip_params = sum(p.numel() for p in model.clip.visual.parameters() if p.requires_grad)
proj_params = sum(p.numel() for p in model.visual_projection.parameters())
print(f'CLIP encoder: {clip_params:,}, Projection: {proj_params:,}')

# Zero-shot benchmark first
model.to(DEVICE).eval()
prompts = ['a photo of recyclable waste', 'a photo of electronic waste', 'a photo of organic waste']
zs_preds, zs_labels = [], []
for x, y in tqdm(val_loader_full, desc='Zero-shot', leave=False):
    logits = model.zero_shot_classify(x.to(DEVICE), prompts)
    zs_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    zs_labels.extend(y.numpy().tolist())
from sklearn.metrics import f1_score
print(f'Zero-shot F1 macro: {f1_score(zs_labels, zs_preds, average="macro"):.4f}')

# Stage 1: Recyclable + Organic, 224px (fine-tune projection only)
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=10, epochs_finetune=15,
    lr_head=1e-3, lr_finetune=1e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=6, epochs_finetune=10,
    lr_head=5e-4, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 — Recalibrate Head (full 3 classes)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate Head @ 224px (full 3 classes) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=8, epochs_finetune=0,
    lr_head=1e-4, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 8.1 TRAIN: Swin-Tiny

In [ ]:
MODEL = 'swin_tiny_patch4_window7_224'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 224px
print(f'\nStage 1: Recyclable + Organic @ 224px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 224px
print(f'\nStage 2: Organic + Electronic @ 224px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Phase 3 — Recalibrate Head (full 3 classes)
print(f'\n=== {MODEL}: Phase 3 — Recalibrate Head @ 224px (full 3 classes) ===')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location='cpu'))
fit(model, train_loader_full, val_loader_full,
    name=f'{MODEL}_final', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=0,
    lr_head=3e-4, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_final')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_final.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 9. Summary Table

In [ ]:
models = ['efficientformer_l1', 'convnext_tiny', 'swin_tiny_patch4_window7_224', 'clip_vit_b32']
rows = []
for m in models:
    path = OUT_DIR / f'{m}.json'
    if path.exists():
        d = json.load(open(path))
        rows.append({
            'model': m,
            'params': f'{sum(p.numel() for p in TrashClassifier(encoder_name=m, num_classes=3).parameters()):,}' if m != 'clip_vit_b32' else '150M+',
            'f1_macro': d['f1_macro'],
            'f1_0_recyclable': d['f1_per_class'][0],
            'f1_1_electronic': d['f1_per_class'][1],
            'f1_2_organic': d['f1_per_class'][2],
            'precision': d['precision'],
            'recall': d['recall'],
        })

df_result = pd.DataFrame(rows).sort_values('f1_macro', ascending=False)
print(df_result.to_string(index=False))

df_result.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f'\nSaved: {OUT_DIR / "summary.csv"}')